# Amazon 2023 Parquet NLP Pipeline (V5)
**Version 5:** Processes manually downloaded `.parquet` files from the Hugging Face Files tab.
Run this notebook in Google Colab AFTER you have uploaded your `.parquet` files to the Colab files section.
**Crucial:** Make sure to select `Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU`.

In [ ]:
!pip install transformers torch pandas scikit-learn tqdm pyarrow fastparquet

In [ ]:
import pandas as pd
import numpy as np
import torch
import glob
from transformers import pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.notebook import tqdm

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================
CATEGORY_NAME = "Electronics"  # Change this to whatever category you are processing today (e.g. "Fashion")
OUTPUT_JSON_PATH = f"{CATEGORY_NAME.lower()}_processed.json"

device = 0 if torch.cuda.is_available() else -1

print("Loading Hugging Face Sentiment Model...")
sentiment_analyzer = pipeline(
    "sentiment-analysis", 
    model="LiYuan/amazon-review-sentiment-analysis", 
    device=device,
    truncation=True,
    max_length=512
)

In [ ]:
# ==========================================
# 2. LOADING MANUAL PARQUET FILES
# ==========================================
print("\n--- Merging Parquet Files ---")
print("Make sure you uploaded your review and meta .parquet files to Colab!")

# Use glob to find all files that start with 'meta_' and end in '.parquet'
meta_files = glob.glob("meta_*.parquet")
# Find all review files (assumes they don't start with meta_)
review_files = glob.glob("*.parquet")
review_files = [f for f in review_files if not f.startswith("meta_")]

if len(meta_files) == 0 or len(review_files) == 0:
    print("\n--- ERROR: No .parquet files found! ---")
    print("Please drag and drop your downloaded files into the Colab file explorer on the left.")
else:
    print(f"Found {len(meta_files)} Metadata files and {len(review_files)} Review files.")

    # Merge all metadata parquet files into one big dataframe
    print("Merging Metadata...")
    meta_df = pd.concat((pd.read_parquet(f) for f in meta_files), ignore_index=True)
    # Standardize column names (McAuley 2023 uses parent_asin)
    meta_df = meta_df.rename(columns={'parent_asin': 'asin', 'average_rating': 'product_star_rating'})
    
    # Sample 400 unique items to keep the script fast 
    items_df = meta_df.drop_duplicates(subset=['asin']).sample(n=min(400, len(meta_df)), random_state=42)
    valid_asins = set(items_df['asin'].tolist())

    print("Merging Reviews and filtering for our 400 items...")
    reviews_df = pd.concat((pd.read_parquet(f) for f in review_files), ignore_index=True)
    reviews_df = reviews_df.rename(columns={'parent_asin': 'asin', 'text': 'body'})
    
    # Only keep reviews for the 400 items we randomly selected
    reviews_df = reviews_df[reviews_df['asin'].isin(valid_asins)]
    
    print(f"\nSuccessfully Loaded: {len(items_df)} Items and {len(reviews_df)} Text Reviews for {CATEGORY_NAME}.")

In [ ]:
# ==========================================
# 3. RUNNING NLP ON ALL DATA
# ==========================================
if len(meta_files) > 0 and len(review_files) > 0:
    print("\n--- Processing Text Reviews with RoBERTa ---")
    reviews_df = reviews_df.dropna(subset=['body'])
    
    # Keep max 30 reviews per product to keep things fast
    sampled_reviews = reviews_df.groupby('asin').apply(lambda x: x.sample(min(len(x), 30))).reset_index(drop=True)

    all_texts = sampled_reviews['body'].astype(str).tolist()

    results = []
    batch_size = 128
    for i in tqdm(range(0, len(all_texts), batch_size)):
        batch = all_texts[i:i+batch_size]
        preds = sentiment_analyzer(batch)
        results.extend(preds)

    sampled_reviews['calculated_sentiment_score'] = [int(r['label'].split()[0]) for r in results]

    review_aggregates = sampled_reviews.groupby('asin').agg({
        'calculated_sentiment_score': 'mean',
        'body': 'count'
    }).rename(columns={'body': 'review_count'}).reset_index()

In [ ]:
# ==========================================
# 4. FEATURE SCORE & HYBRID CALCULATION
# ==========================================
if len(meta_files) > 0 and len(review_files) > 0:
    print("\n--- Merging Data & Calculating Initial Ranks ---")
    final_df = pd.merge(items_df, review_aggregates, on='asin', how='left')

    final_df['title'] = final_df['title'].fillna('')
    final_df['brand'] = final_df['brand'].fillna('Unknown')
    final_df['category'] = CATEGORY_NAME
    final_df['feature_text'] = final_df['brand'] + " " + final_df['category'] + " " + final_df['title']

    tfidf = TfidfVectorizer(stop_words='english', max_features=1000)
    tfidf_matrix = tfidf.fit_transform(final_df['feature_text'])

    doc_lengths = tfidf_matrix.sum(axis=1)
    min_len, max_len = doc_lengths.min(), doc_lengths.max()
    if max_len > min_len:
        final_df['feature_score'] = 1 + 4 * ((doc_lengths - min_len) / (max_len - min_len))
    else:
        final_df['feature_score'] = 3.0

    final_df['feature_score'] = np.asarray(final_df['feature_score']).flatten()

    def calculate_initial_rank(row):
        feat_score = row['feature_score']
        sentiment = row['calculated_sentiment_score']
        static_stars = row['product_star_rating']
        
        has_sentiment = pd.notna(sentiment)
        has_static_stars = pd.notna(static_stars)
        
        if has_sentiment and has_static_stars:
            return (0.6 * sentiment) + (0.2 * static_stars) + (0.2 * feat_score)
        elif has_sentiment and not has_static_stars:
            return (0.7 * sentiment) + (0.3 * feat_score)
        elif not has_sentiment and has_static_stars:
            return (0.7 * static_stars) + (0.3 * feat_score)
        else:
            return feat_score

    final_df['initial_rank'] = final_df.apply(calculate_initial_rank, axis=1)

    # Clean up output
    final_df['calculated_sentiment_score'] = final_df['calculated_sentiment_score'].fillna(0.0).round(2)
    final_df['product_star_rating'] = final_df['product_star_rating'].fillna(0.0).round(2)
    final_df['review_count'] = final_df['review_count'].fillna(0).astype(int)
    final_df['feature_score'] = final_df['feature_score'].round(2)
    final_df['initial_rank'] = final_df['initial_rank'].round(2)
    final_df['price'] = final_df['price'].fillna(0.0)
    
    # Extra: Safely parse McAuley 2023 image arrays
    def extract_image(img_data):
        try:
            if isinstance(img_data, list) and len(img_data) > 0 and isinstance(img_data[0], dict):
                return img_data[0].get('large_image_url', '')
        except:
            pass
        return ''
    final_df['image'] = final_df['images'].apply(extract_image) if 'images' in final_df.columns else ''

    export_df = final_df[['asin', 'title', 'brand', 'category', 'image', 'price', 
                          'product_star_rating', 'review_count', 'calculated_sentiment_score', 
                          'feature_score', 'initial_rank']]

    export_df = export_df.rename(columns={'asin': 'id', 'product_star_rating': 'rating'})
    export_df = export_df.sort_values(by='initial_rank', ascending=False)

In [ ]:
# ==========================================
# 5. EXPORT DATA
# ==========================================
if len(meta_files) > 0 and len(review_files) > 0:
    print(f"\nExporting exactly {len(export_df)} items to JSON...")
    export_df.to_json(OUTPUT_JSON_PATH, orient='records', indent=4)
    print(f"Pipeline Complete! file {OUTPUT_JSON_PATH} has been created.")
    
    try:
        from google.colab import files
        files.download(OUTPUT_JSON_PATH)
    except ImportError:
        print("Not running in Colab. File saved locally.")